# West-Med eOdyn currents through a day

Fetch eOdyn's **OSmose L4** surface currents over the western Mediterranean
(lon -6 to 20, lat 35 to 45) for one day, and watch how they evolve across its
eight 3-hourly analyses, together with their confidence index.

collekt reads OSmose from eOdyn's Ocean Data Access (ODA) API through the bundled
`eodyn_osmose_currents` dataset. Downloading needs ODA credentials: set
`ODA_USERNAME` / `ODA_PASSWORD` in your shell or in a `.env` at the repository
root. Files are written under `notebooks/cache/` (git-ignored); the first run
downloads about 70 MB (eight full-domain files), later runs reuse the cache.

In [ ]:
from pathlib import Path

import collekt


def _repo_root(marker: str = "pyproject.toml") -> Path:
    """Anchor the cache to the repo root, whatever the kernel's working dir."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / marker).exists():
            return candidate
    return here


OUTPUT_ROOT = _repo_root() / "notebooks" / "cache" / "westmed_eodyn"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

REGION = collekt.Region.from_bbox((-6.0, 20.0, 35.0, 45.0))

# Total current components and their confidence index; the speed is computed below.
config = collekt.DatasetConfig(
    collekt.eOdyn("eodyn_osmose_currents", variables=["totalewct", "totalnsct", "ci"]),
)
config.as_dict()

## Pick the day

OSmose files reach ODA one to two days after their analysis time, so yesterday may
not be published yet. The dry run reads the dataset's live coverage from ODA: we
take yesterday, or the latest published day if yesterday is not there yet. Set
`DAY` to an ISO date to choose another day.

In [ ]:
from datetime import UTC, date, datetime, timedelta

DAY = None  # e.g. "2026-11-21"; None picks yesterday, or the latest day on ODA


def make_fetcher(day):
    request = collekt.Request(region=REGION, start=day, end=day, metadata={"region_name": "west-med"})
    return collekt.Fetcher(
        request=request,
        config=config,
        output_dir=OUTPUT_ROOT,
        progress=lambda source, message: print(f"[{source}] {message}"),
    )


if DAY is None:
    yesterday = datetime.now(UTC).date() - timedelta(days=1)
    availability = make_fetcher(yesterday.isoformat()).plan().results[0].details["availability"]
    coverage = availability["coverage"]
    print(f"ODA coverage: {coverage['start']} to {coverage['end']} (method: {availability['method']})")
    DAY = min(yesterday, date.fromisoformat(coverage["end"])).isoformat()

print(f"Day: {DAY}")
fetcher = make_fetcher(DAY)
fetcher.plan()

## Download

One NetCDF for the day, holding the eight 3-hourly analyses cropped to the region.

In [ ]:
result = fetcher.download()
if not result.files:
    raise RuntimeError(result.results[0].message)
result

In [ ]:
import numpy as np
import xarray as xr

ds = xr.open_dataset(result.files[0]).load()
ds["speed"] = np.hypot(ds.totalewct, ds.totalnsct)
ds["speed"].attrs = {"long_name": "total surface current speed", "units": "m s-1"}
ds

## Currents every 3 hours

Each panel is one analysis: colour is the total current speed, arrows its
direction. All panels share one colour scale, so they compare directly.

In [ ]:
import math

import matplotlib.pyplot as plt

ARROW_STEP = 12  # one arrow every 12 grid points (0.6°)
arrows = ds.isel(latitude=slice(None, None, ARROW_STEP), longitude=slice(None, None, ARROW_STEP))
vmax = float(ds.speed.quantile(0.99))
aspect = 1 / np.cos(np.deg2rad(float(ds.latitude.mean())))  # equal distances in both directions


def draw(ax, index):
    """Draw the speed field and arrows of one time step; return the artists."""
    step, arrow = ds.isel(time=index), arrows.isel(time=index)
    mesh = ax.pcolormesh(ds.longitude, ds.latitude, step.speed, vmin=0, vmax=vmax, cmap="viridis")
    quiver = ax.quiver(
        arrow.longitude, arrow.latitude, arrow.totalewct, arrow.totalnsct, color="white", scale=12, width=0.0025
    )
    ax.set_title(f"{np.datetime_as_string(step.time.values, unit='m')} UTC")
    ax.set_aspect(aspect)
    return mesh, quiver


steps = ds.sizes["time"]


def panel_grid():
    """One panel per time step, two per row; return the figure and the used axes."""
    rows = math.ceil(steps / 2)
    fig, axes = plt.subplots(rows, 2, figsize=(14, 3.6 * rows), sharex=True, sharey=True, layout="constrained")
    for ax in axes.ravel()[steps:]:
        ax.set_visible(False)
    return fig, axes.ravel()[:steps]


fig, axes = panel_grid()
for index, ax in enumerate(axes):
    mesh, quiver = draw(ax, index)
ax.quiverkey(quiver, 0.36, 0.06, 0.5, "0.5 m/s", labelpos="E", coordinates="axes", color="black")  # over Algeria
fig.colorbar(mesh, ax=axes, shrink=0.6, label="total surface current speed (m/s)")
fig.suptitle(f"OSmose L4 total surface currents, {DAY}")
plt.show()

## What changes through the day

Much of the flow (the geostrophic part) barely moves in 3 hours. Subtracting the
daily mean leaves what does change: each panel shows one analysis minus the day's
mean current, as the speed of that difference with arrows for its direction.

In [ ]:
anomaly = ds[["totalewct", "totalnsct"]] - ds[["totalewct", "totalnsct"]].mean("time")
anomaly["speed"] = np.hypot(anomaly.totalewct, anomaly.totalnsct)
anomaly_arrows = anomaly.isel(latitude=slice(None, None, ARROW_STEP), longitude=slice(None, None, ARROW_STEP))
anomaly_max = float(anomaly.speed.quantile(0.99))

fig, axes = panel_grid()
for index, ax in enumerate(axes):
    step, arrow = anomaly.isel(time=index), anomaly_arrows.isel(time=index)
    mesh = ax.pcolormesh(ds.longitude, ds.latitude, step.speed, vmin=0, vmax=anomaly_max, cmap="magma")
    ax.quiver(
        arrow.longitude, arrow.latitude, arrow.totalewct, arrow.totalnsct, color="tab:cyan", scale=2, width=0.0025
    )
    ax.set_title(f"{np.datetime_as_string(ds.time.values[index], unit='m')} UTC minus daily mean")
    ax.set_aspect(aspect)
fig.colorbar(mesh, ax=axes, shrink=0.6, label="departure from the daily mean current (m/s)")
plt.show()

## Confidence index

`ci` is OSmose's velocity confidence index, from 0 (unreliable) to 1 (reliable),
derived from the posterior uncertainty of the current estimate. It shows where the
currents above are well constrained, and how that changes through the day.

In [ ]:
fig, axes = panel_grid()
for index, ax in enumerate(axes):
    step = ds.isel(time=index)
    mesh = ax.pcolormesh(ds.longitude, ds.latitude, step.ci, vmin=0, vmax=1, cmap="cividis")
    ax.set_title(f"{np.datetime_as_string(step.time.values, unit='m')} UTC")
    ax.set_aspect(aspect)
fig.colorbar(mesh, ax=axes, shrink=0.6, label="velocity confidence index")
fig.suptitle(f"OSmose L4 confidence index, {DAY}")
plt.show()

## Animate the day

Step through the analyses to see the currents move. Use the player's controls to
pause or loop.

In [ ]:
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

fig, ax = plt.subplots(figsize=(11, 5.5), layout="constrained")
mesh, quiver = draw(ax, 0)
fig.colorbar(mesh, ax=ax, label="total surface current speed (m/s)")


def update(index):
    step, arrow = ds.isel(time=index), arrows.isel(time=index)
    mesh.set_array(np.ma.masked_invalid(step.speed.values))
    quiver.set_UVC(arrow.totalewct.values, arrow.totalnsct.values)
    ax.set_title(f"{np.datetime_as_string(step.time.values, unit='m')} UTC")
    return mesh, quiver


animation = FuncAnimation(fig, update, frames=steps, interval=700)
plt.close(fig)
HTML(animation.to_jshtml())